# Comparando pandas (CPU) vs cuDF (GPU) para cargas de trabajo con DataFrames


En este notebook, comparamos pandas ejecutándose en la CPU con RAPIDS cuDF ejecutándose en la GPU para operaciones comunes sobre DataFrames como carga de datos, filtrado, agrupamiento y unión.

Nos enfocamos en:

- Diferencias en el tiempo de ejecución entre CPU y GPU.
- Similitudes y diferencias en las APIs.
- Consideraciones prácticas como el tamaño del dataset y el uso de memoria de la GPU.

## 1. Entorno y configuración


Esta sección verifica las versiones de Python, pandas y cuDF, y comprueba que haya una GPU disponible.

In [1]:
import os
import sys
import subprocess
import time
import numpy as np
import pandas as pd

In [2]:
# Opcional: Configuración de cuDF para Google Colab y Visual Studio Code

def install_cudf():
    try:
        import cudf
        print("cuDF ya está instalado:", cudf.__version__)
        return True
    except ImportError:
        print("cuDF no encontrado.")
        print("Intentando instalar...")

    # Caso: Google Colab
    in_colab = "google.colab" in sys.modules
    if in_colab:
        print("Entorno detectado: Google Colab")
        try:
            import torch
            if not torch.cuda.is_available():
                print("No se detectó una GPU activa.")
                print("Por favor habilita la GPU en: Runtime > Change runtime type")
                return False
        except:
            pass
        # Instalación oficial de RAPIDS para Colab
        !pip install -q cudf-cu12 --extra-index-url=https://pypi.nvidia.com

    # Caso: Entorno local (VS Code u otros)
    else:
        print("Entorno detectado: Local (VS Code u otro)")
        print("cuDF requiere instalación manual con RAPIDS.")
        print("Pasos sugeridos:")
        print("  conda create -n rapids python=3.10")
        print("  conda activate rapids")
        print("  pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com")
        print("Alternativa (venv en lugar de conda):")
        print("Advertencia: Requiere versión de Python compatible (3.10 o 3.11) y entorno Linux")
        print("Paso 1: Instalar Python 3.10 o 3.11 (si no está instalado)")
        print("Paso 2: Crear entorno virtual con esa versión:")
        print("  python3.10 -m venv rapids_env   # o python3.11")
        print("Paso 3: Activar el entorno:")
        print("  source rapids_env/bin/activate   # Linux / Mac")
        print("  rapids_env\\Scripts\\activate.bat  # Windows (CMD)")
        print("Paso 4: Actualizar pip:")
        print("  python -m pip install --upgrade pip")
        print("Paso 5: Instalar cuDF:")
        print("  pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com")
        print("Nota: cuDF está principalmente soportado en Linux. En Windows puede fallar.")
        print("Nota: Para mejores resultados, usa Google Colab o WSL2.")
        return False

    # Verificación de la instalación
    try:
        import cudf
        print("cuDF instalado correctamente:", cudf.__version__)
        return True
    except ImportError:
        print("Error al instalar cuDF.")
        return False

CUDF_AVAILABLE = install_cudf()

cuDF ya está instalado: 26.02.01


In [3]:
# Verificación de versiones

print("Versión de Python:", sys.version)
print("Versión de pandas:", pd.__version__)

try:
    import cudf
    CUDF_AVAILABLE = True
    print("Versión de cuDF:", cudf.__version__)
except ImportError:
    CUDF_AVAILABLE = False
    print("cuDF no está instalado. Se omitirán los benchmarks en GPU.")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

Versión de Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Versión de pandas: 2.2.2
Versión de cuDF: 26.02.01


In [4]:
# Opcional: Comprobar información de la GPU usando nvidia-smi (si está disponible)

!nvidia-smi || echo "nvidia-smi no está disponible en este sistema"

Fri Mar 20 12:11:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Experimento

En ciencia de datos en Python, pandas es una de las librerías más utilizadas para trabajar con datos tabulares mediante estructuras llamadas DataFrames. Estas permiten filtrar, agrupar, transformar y analizar datos de forma eficiente. Sin embargo, pandas ejecuta sus operaciones en la CPU (Central Processing Unit), es decir, el procesador principal del computador, que está optimizado para tareas generales y secuenciales.

Por otro lado, cuDF es una librería desarrollada dentro del ecosistema RAPIDS, que ofrece una API muy similar a pandas, pero ejecuta las operaciones en la GPU (Graphics Processing Unit). A diferencia de la CPU, la GPU está diseñada para realizar muchas operaciones en paralelo, lo que la hace especialmente eficiente para procesar grandes volúmenes de datos.

El proyecto RAPIDS (desarrollado por NVIDIA) busca precisamente acelerar flujos de trabajo de ciencia de datos utilizando GPUs, permitiendo que muchas operaciones comunes (como filtrado, groupby o joins) se ejecuten mucho más rápido en comparación con su equivalente en CPU.

En este experimento se propone:

1. Generar un dataset sintético con varios millones de filas.
2. Ejecutar las mismas operaciones con pandas (CPU).
3. Ejecutar las mismas operaciones con cuDF (GPU).
4. Medir el tiempo de ejecución usando `time.perf_counter`.
5. Comparar los resultados y analizar cuándo la aceleración con GPU es beneficiosa.

### Generación de un dataset sintético

In [5]:
# Tamaño configurable según la RAM / GPU disponible
N_ROWS = 5_000_000

categories = ["A", "B", "C", "D", "E"]
n_categories = len(categories)

print(f"Generando dataset sintético con {N_ROWS:,} filas...")

df_cpu = pd.DataFrame({
    "id": np.arange(N_ROWS, dtype=np.int64),
    "category": np.random.choice(categories, size=N_ROWS),
    "value1": np.random.randn(N_ROWS).astype(np.float32),
    "value2": np.random.randint(0, 1000, size=N_ROWS, dtype=np.int32),
})

# Guardar dataset en CSV (para simular carga de trabajo de I/O real)
csv_path = "synthetic_dataset.csv"
df_cpu.to_csv(csv_path, index=False)

# Información del tamaño del archivo
file_size_mb = os.path.getsize(csv_path) / 1e6
print(f"Dataset guardado en {csv_path} ({file_size_mb:.2f} MB)")

Generando dataset sintético con 5,000,000 filas...
Dataset guardado en synthetic_dataset.csv (122.88 MB)


### Función auxiliar para benchmarking

Para poder comparar de forma justa el rendimiento entre CPU y GPU, necesitamos medir cuánto tiempo tarda cada operación.

En lugar de repetir el mismo código de medición una y otra vez, encapsulamos este comportamiento en una función auxiliar. Esto nos permite reutilizarla fácilmente y mantener el código más limpio y consistente.

La idea es simple:
- Registrar el tiempo antes de ejecutar una función.
- Ejecutar la operación que queremos medir.
- Registrar el tiempo después.
- Calcular la diferencia.

In [6]:
# func: Función de interés
# *args: Todos los argumentos posicionales que se le pasarán a func
# **kwargs: Todos los keyword arguments que se le pasarán a func

def benchmark(func, *args, **kwargs):
    start = time.perf_counter()
    result = func(*args, **kwargs)
    end = time.perf_counter()
    return result, end - start

### Baseline: pandas (CPU)

En esta sección ejecutamos todas las operaciones utilizando pandas, es decir, trabajando únicamente sobre la CPU. Esto nos servirá como referencia (baseline) para luego comparar con la versión en GPU usando cuDF.

Las operaciones que evaluaremos son típicas en flujos de trabajo de análisis de datos:

- Carga de datos desde disco (CSV).
- Filtrado de filas según condiciones.
- Agrupación (groupby) con agregaciones.
- Unión de datos (join/merge) con otra tabla.

**Nota:** Cada una de estas operaciones se ejecuta utilizando la función de benchmarking, que retorna una tupla del tipo (resultado, tiempo de ejecución). Esto permite, por un lado, obtener el resultado de la operación (por ejemplo, un DataFrame transformado) y, por otro, medir cuánto tiempo tardó en ejecutarse. De esta forma, podemos analizar el rendimiento sin perder acceso a los datos generados en cada paso.

In [7]:
print("=== pandas (CPU) benchmarks ===")

# 1. Cargar CSV
df_pandas, t_load_pandas = benchmark(pd.read_csv, csv_path)
print(f"pandas - load CSV: {t_load_pandas:.3f} seconds")

# 2. Filtrado simple
def pandas_filter(df):
    return df[(df["value1"] > 0) & (df["value2"] > 500)]

filtered_pandas, t_filter_pandas = benchmark(pandas_filter, df_pandas)
print(f"pandas - filter: {t_filter_pandas:.3f} seconds")

# 3. Agrupación Groupby
def pandas_groupby(df):
    return df.groupby("category")["value1"].agg(["mean", "std", "count"])

grouped_pandas, t_group_pandas = benchmark(pandas_groupby, df_pandas)
print(f"pandas - groupby: {t_group_pandas:.3f} seconds")

# 4. Join / merge
def pandas_join(df):
    lookup = pd.DataFrame({
        "category": categories,
        "weight": np.linspace(1.0, 2.0, n_categories).astype(np.float32)
    })
    return df.merge(lookup, on="category", how="left")

joined_pandas, t_join_pandas = benchmark(pandas_join, df_pandas)
print(f"pandas - join: {t_join_pandas:.3f} seconds")

=== pandas (CPU) benchmarks ===
pandas - load CSV: 1.557 seconds
pandas - filter: 0.080 seconds
pandas - groupby: 0.354 seconds
pandas - join: 0.559 seconds


#### Detalle `pandas_groupby`

Esta función toma un DataFrame y realiza una agregación agrupando por una categoría.


In [8]:
# Función
def pandas_groupby(df):
    return df.groupby("category")["value1"].agg(["mean", "std", "count"])

Explicación paso a paso:

1. `groupby("category")`:
    - Agrupa las filas del DataFrame según los valores únicos de la columna "category".
    - Por ejemplo, si category tiene valores ["A", "B", "C"], se crean 3 grupos.

2. `["value1"]`:
    - Seleccionamos la columna sobre la cual queremos calcular estadísticas.
    - Es decir, ignoramos otras columnas como "value2".

3. `agg(["mean", "std", "count"])`:
    - Aplicamos múltiples funciones de agregación sobre cada grupo:
     - mean: promedio de los valores
     - std: desviación estándar (qué tan dispersos están los datos)
     - count: cantidad de filas en el grupo

4. El resultado es un nuevo DataFrame donde:
    - Cada fila representa una categoría
    - Cada columna representa una métrica calculada


In [9]:
# Ejecución de ejemplo
grouped_example = pandas_groupby(df_pandas)
print(grouped_example.head())

              mean       std    count
category                             
A         0.002027  1.000471   998453
B        -0.000050  0.999947  1000392
C         0.000749  0.999491  1000503
D        -0.001385  1.000060  1000446
E         0.000693  1.001672  1000206


#### Detalle `pandas_join`

Esta función combina el DataFrame original con una tabla auxiliar (lookup).

In [10]:
# Función
def pandas_join(df):
    # Tabla de referencia (lookup table)
    lookup = pd.DataFrame({
        "category": categories,
        "weight": np.linspace(1.0, 2.0, n_categories).astype(np.float32)
    })
    return df.merge(lookup, on="category", how="left")

Explicación paso a paso

1. `lookup` (tabla de referencia):
    - Es un DataFrame pequeño que contiene información adicional.
    - En este caso, simplemente asigna un "peso" (weight) a cada categoría.

2. `merge(...)`:
    - Es la función de pandas para combinar dos DataFrames.
    - Similar a un JOIN en SQL.

3. Parámetros clave:
    - `on="category"`: Indica la columna común que se usará para hacer la unión.
    - `how="left"`: Tipo de join que mantiene todas las filas del DataFrame original.



In [11]:
# Ejecución de ejemplo
joined_example = pandas_join(df_pandas)
print(joined_example.head())

   id category    value1  value2  weight
0   0        D -0.957101     715    1.75
1   1        E  2.132025     990    2.00
2   2        C -0.022485     400    1.50
3   3        E  0.196817     744    2.00
4   4        E -0.369941     352    2.00


**Actividad:** Investiga sobre los diferentes tipos de Join y compara sus casos de uso.

### Benchmarking para cuDF (GPU)

Ahora, repetimos las mismas operaciones utilizando cuDF sobre la GPU.

In [12]:
if not CUDF_AVAILABLE:
    print("cuDF not available: skipping GPU benchmarks.")
else:
    import cudf

    print("=== cuDF (GPU) benchmarks ===")

    # 1. Cargar CSV
    gdf, t_load_cudf = benchmark(cudf.read_csv, csv_path)
    print(f"cuDF - load CSV: {t_load_cudf:.3f} seconds")

    # 2. Filtrado simple
    def cudf_filter(gdf_):
        return gdf_[(gdf_["value1"] > 0) & (gdf_["value2"] > 500)]

    filtered_cudf, t_filter_cudf = benchmark(cudf_filter, gdf)
    print(f"cuDF - filter: {t_filter_cudf:.3f} seconds")

    # 3. Agrupación Groupby
    def cudf_groupby(gdf_):
        return gdf_.groupby("category")["value1"].agg(["mean", "std", "count"])

    grouped_cudf, t_group_cudf = benchmark(cudf_groupby, gdf)
    print(f"cuDF - groupby: {t_group_cudf:.3f} seconds")

    # 4. Join / merge (no CuPy needed; use NumPy array directly)
    def cudf_join(gdf_):
        weights_np = np.linspace(1.0, 2.0, n_categories).astype(np.float32)
        lookup_gdf = cudf.DataFrame({
            "category": categories,
            "weight": weights_np,
        })
        return gdf_.merge(lookup_gdf, on="category", how="left")

    joined_cudf, t_join_cudf = benchmark(cudf_join, gdf)
    print(f"cuDF - join: {t_join_cudf:.3f} seconds")

=== cuDF (GPU) benchmarks ===
cuDF - load CSV: 0.898 seconds
cuDF - filter: 0.121 seconds
cuDF - groupby: 0.160 seconds
cuDF - join: 0.167 seconds


#### Detalle `cudf_join`

La función cudf_join realiza un join entre un DataFrame en GPU (cuDF) y una tabla de referencia también en formato cuDF.

In [13]:
# Función
def cudf_join(gdf_):
    # Generamos los pesos como un arreglo de NumPy (en CPU)
    weights_np = np.linspace(1.0, 2.0, n_categories).astype(np.float32)
    # Creamos un DataFrame cuDF (en GPU) a partir de datos en CPU
    lookup_gdf = cudf.DataFrame({
        "category": categories,
        "weight": weights_np,
    })
    # Realizamos el merge en GPU
    return gdf_.merge(lookup_gdf, on="category", how="left")

In [14]:
# Ejecución de ejemplo
joined_gpu_example = cudf_join(gdf)
print(joined_gpu_example.head())

     id category    value1  value2  weight
0  2208        E -1.047718     729    2.00
1  2209        E  0.438955     509    2.00
2  2210        B  0.285953      43    1.25
3  2211        B  1.027694     287    1.25
4  2212        E  0.091374     258    2.00


### Tabla resumen de resultados


Se recopilan todos los tiempos de ejecución en una única tabla para comparar directamente pandas (CPU) y cuDF (GPU).

In [15]:
import math

results = []

# Resultados Pandas
results.append(("load_csv", "pandas", t_load_pandas))
results.append(("filter", "pandas", t_filter_pandas))
results.append(("groupby", "pandas", t_group_pandas))
results.append(("join", "pandas", t_join_pandas))

# Resultados cuDF (si están disponibles)
if CUDF_AVAILABLE:
    results.append(("load_csv", "cuDF", t_load_cudf))
    results.append(("filter", "cuDF", t_filter_cudf))
    results.append(("groupby", "cuDF", t_group_cudf))
    results.append(("join", "cuDF", t_join_cudf))

results_df = pd.DataFrame(results, columns=["operation", "library", "time_sec"])
results_pivot = results_df.pivot(index="operation", columns="library", values="time_sec")

if CUDF_AVAILABLE:
    results_pivot["speedup_cuDF_vs_pandas"] = results_pivot["pandas"] / results_pivot["cuDF"]

results_pivot

library,cuDF,pandas,speedup_cuDF_vs_pandas
operation,,,
filter,0.120701,0.080206,0.664502
groupby,0.160106,0.354458,2.213900
join,0.166596,0.558624,3.353173
load_csv,0.898082,1.557485,1.734235


La tabla anterior resume el tiempo de ejecución de cada operación utilizando pandas (CPU) y cuDF (GPU), además del factor de aceleración (speedup) cuando cuDF está disponible.

## 3. Conclusiones

### Observaciones relevantes:



- En datasets pequeños, el costo de transferir datos entre CPU y GPU puede reducir significativamente, o incluso anular, los beneficios de utilizar la GPU.

- En datasets grandes (millones de filas), cuDF puede ofrecer mejoras de rendimiento importantes, especialmente en operaciones como groupby y join, debido al alto grado de paralelismo y al ancho de banda de memoria de la GPU.

- La similitud entre la API de pandas y cuDF facilita la migración de código, pero no todas las funcionalidades de pandas están soportadas. En particular, operaciones que dependen de objetos arbitrarios de Python o funciones complejas con .apply pueden no estar disponibles o no estar optimizadas en GPU.

- El rendimiento observado depende no solo del tipo de operación, sino también del flujo completo de datos. Factores como la cantidad de transferencias entre CPU y GPU, el tamaño del dataset y la naturaleza de las transformaciones influyen directamente en los resultados.

- Las operaciones vectorizadas y estructuradas (como filtros, agregaciones y joins) tienden a beneficiarse más del uso de GPU, mientras que operaciones más secuenciales o difíciles de paralelizar pueden no mostrar mejoras significativas.

- La disponibilidad de memoria en la GPU puede convertirse en una limitación en comparación con la memoria RAM del sistema, especialmente en datasets muy grandes.

En conjunto, estos resultados permiten entender que el uso de GPU no siempre implica una mejora automática de rendimiento, sino que depende del contexto, el tamaño de los datos y el tipo de operaciones realizadas.

### Similitudes y diferencias en la API


- cuDF ofrece una API de DataFrames diseñada para ser intencionalmente similar a pandas (`DataFrame`, `Series`, `Index`, `read_csv`, `merge`, `groupby`, etc.). Esto permite que muchos flujos de trabajo existentes puedan adaptarse a GPU con cambios mínimos en el código.

- Sin embargo, cuDF no es un reemplazo completamente transparente (drop-in replacement) de pandas en todos los casos:
  - cuDF no soporta objetos arbitrarios de Python dentro de las columnas. Aunque las columnas de texto existen, están implementadas mediante estructuras optimizadas para GPU y suelen reportarse como tipo `object`, pero no tienen la misma flexibilidad que en pandas.
  - Operaciones como `.apply` y otras que dependen de funciones definidas por el usuario tienen restricciones. En cuDF, estas funciones deben ser compatibles con ejecución en GPU (device functions), lo que limita el uso de ciertas construcciones de Python (por ejemplo, estructuras complejas o dependencias externas).

- Versiones más recientes introducen un modo acelerador de pandas (`cudf.pandas`), cuyo objetivo es ofrecer una mayor compatibilidad con la API de pandas, ejecutando operaciones en GPU de forma transparente cuando es posible.

En general, cuDF destaca especialmente en datasets grandes, numéricos o estructurados, donde las operaciones pueden vectorizarse y paralelizarse eficientemente. Por su parte, pandas sigue siendo una herramienta altamente flexible, adecuada para datasets pequeños o medianos, así como para casos donde se requiere mayor libertad en el uso de lógica personalizada o cuando no se dispone de GPU.

### Consideraciones prácticas: CPU vs GPU


Al decidir entre pandas (CPU) y cuDF (GPU), es importante considerar varios factores:

- **Tamaño del dataset:** Las GPUs muestran su mayor ventaja en datasets medianos a grandes, donde el paralelismo masivo compensa el costo de transferir datos desde la CPU. En datasets pequeños, este costo puede dominar el tiempo total.

- **Memoria disponible:** La memoria de la GPU suele ser significativamente menor que la memoria RAM del sistema. Esto implica que datasets muy grandes pueden no caber completamente en GPU, requiriendo técnicas como unified memory o procesamiento por bloques (out-of-core).

- **Tipo de operaciones:** Operaciones vectorizadas y bien estructuradas (filtrado, agregaciones, joins) se benefician fuertemente del paralelismo de la GPU. En contraste, operaciones con lógica altamente personalizada, iterativa o difícil de paralelizar suelen ejecutarse de forma más eficiente en CPU.

- **Costo de transferencia de datos:** Mover datos entre CPU y GPU no es gratuito. Si el flujo de trabajo implica múltiples transferencias, el beneficio de la GPU puede verse reducido.

- **Infraestructura y requisitos:** cuDF requiere una GPU NVIDIA compatible con CUDA y un entorno correctamente configurado. En cambio, pandas funciona en prácticamente cualquier sistema con CPU, lo que lo hace más accesible y portable.

- **Madurez del ecosistema:** pandas cuenta con un ecosistema muy amplio y consolidado, con gran cantidad de herramientas y extensiones.
cuDF, aunque en constante evolución, puede presentar limitaciones en funcionalidades específicas o menor compatibilidad con ciertas librerías.


## 4. Ejercicio: Predicción y comparación CPU vs GPU

En este ejercicio se busca analizar un nuevo dataset, formular hipótesis sobre el rendimiento esperado en CPU y GPU, y luego validar dichas hipótesis mediante benchmarking.

### Generación del dataset

Se generará un nuevo dataset sintético con características ligeramente distintas al anterior, introduciendo mayor cardinalidad y mayor número de columnas.

In [16]:
N_ROWS = 3_000_000

regions = ["North", "South", "East", "West"]
products = [f"P{i}" for i in range(20)]  # mayor cardinalidad

print(f"Generando dataset con {N_ROWS:,} filas...")

df_new = pd.DataFrame({
    "id": np.arange(N_ROWS, dtype=np.int64),
    "region": np.random.choice(regions, size=N_ROWS),
    "product": np.random.choice(products, size=N_ROWS),
    "sales": np.random.gamma(shape=2.0, scale=50.0, size=N_ROWS).astype(np.float32),
    "quantity": np.random.randint(1, 20, size=N_ROWS, dtype=np.int32),
})

csv_new_path = "synthetic_sales.csv"
df_new.to_csv(csv_new_path, index=False)

print("Dataset generado.")

Generando dataset con 3,000,000 filas...
Dataset generado.


### Función auxiliar para benchmarking

Reutilizamos la del experimento anterior.

In [17]:
def benchmark(func, *args, **kwargs):
    start = time.perf_counter()
    result = func(*args, **kwargs)
    end = time.perf_counter()
    return result, end - start

### Definición de operaciones

Implementa una serie de operaciones típicas sobre el dataset, tanto en pandas (CPU) como en cuDF (GPU). Replica exactamente las mismas transformaciones en ambos casos para poder comparar su rendimiento posteriormente.

Implementa las siguientes operaciones:

- Carga de datos: Leer el archivo CSV generado previamente y almacenarlo en un DataFrame.

- Filtrado: Seleccionar únicamente las filas donde `sales > 50` y `quantity > 5`.

- Agregación: Agrupar los datos por la columna `product`, calculando las métricas promedio, suma y conteo sobre `sales`.

- Unión: Crear una tabla auxiliar con la columna `product` y una columna `price_factor` con valores entre 0.8 y 1.2 para luego realizar un left join con el dataset original usando la columna `product`.

In [18]:
# TODO pandas

In [19]:
# TODO cuDF

### Predicciones previas al benchmarking

Antes de ejecutar cualquier medición de rendimiento, es importante formular hipótesis sobre el comportamiento esperado. El objetivo de esta sección no es “acertar”, sino razonar en base a las características del dataset y las operaciones definidas.

Redacta al menos tres hipótesis considerando aspectos como:

- Tamaño del dataset
- Tipo de operaciones (filtrado, agregación, join)
- Nivel de paralelismo posible
- Costos de transferencia de datos

-------------------

Adicionalmente, responde brevemente las siguientes preguntas, justificando tus respuestas:

- ¿Qué operación crees que será más rápida en GPU en comparación con CPU? ¿Por qué?

- ¿Qué operación crees que tendrá menor diferencia entre CPU y GPU?

- ¿Cómo influye la cardinalidad de product en la operación de groupby?

- ¿Crees que el join será más costoso que en el dataset anterior? ¿Por qué?

- En este caso, ¿esperas que usar GPU valga la pena en general?
Justifica considerando el tamaño del dataset y el tipo de operaciones.

### Benchmark en pandas (CPU)

Utilizando las funciones definidas previamente, mide el tiempo de ejecución de cada operación mediante la función benchmark.

In [20]:
# TODO

### Benchmark en cuDF (GPU)

Repetir exactamente el mismo proceso anterior, pero utilizando las funciones equivalentes en cuDF.

In [21]:
# TODO

### Análisis de resultados

Organiza los resultados obtenidos y compáralos con las predicciones realizadas previamente. Para esto, construye una tabla que resuma los tiempos de ejecución en CPU (pandas) y GPU (cuDF), junto con el speedup cuando sea posible.

In [22]:
# TODO

--------------------

Adicionalmente, responde a estas preguntas:

- ¿Se cumplieron las predicciones realizadas previamente? Explicar en qué casos sí y en cuáles no.

- ¿Qué operación mostró mayor aceleración en GPU? ¿Por qué esta operación se beneficia más del paralelismo?

- ¿Qué operación mostró menor diferencia entre CPU y GPU? Relacionar con su complejidad y costo computacional.

- ¿Qué rol crees que juega el tamaño del dataset en estos resultados? ¿Y los tipos de operación?